# KoMA-RAG ablation study (Kaggle runner)

Runs `run_ablation.py` on Kaggle. Before running:

1. **Settings (right sidebar) -> Internet: On** (required to reach the GitHub repo, PyPI/uv's Python downloads, and the Mistral API).
2. **Add-ons -> Secrets -> Add a new secret**, name it `MISTRAL_API_KEY`, paste your Mistral key, attach it to this notebook.
3. Push your local fixes to GitHub first -- this notebook clones from `origin`, branch `KoMA-V2`. If you haven't pushed yet, `git clone` below will pull the *old* code without the latest fixes.
4. No GPU needed -- everything here runs on CPU, so leave the accelerator off and save your GPU-hours quota.

**Python version:** Kaggle's kernel Python is 3.12+, but this repo's pinned dependencies (numpy 1.24.3, gymnasium 0.28.1) only have prebuilt wheels up to Python 3.11 and fail to build from source on 3.12 (confirmed by an actual run -- every phase crashed in ~2s from a broken numpy install). Cell 2 works around this with `uv`: it downloads an isolated Python 3.9 into `/kaggle/temp/venv` and installs everything there instead of into the kernel's own environment. Every later cell runs the ablation script through that venv (`VENV_PY`, set in cell 1), not through plain `python`/`pip`.

**Session length:** Kaggle notebook sessions have a runtime cap (historically ~9-12h; check your current limit under Settings). At Mistral's ~7s/call, the full batch (10 seed + 20 episodes x 4 configs, full 20-step episodes) is roughly 3.5-7h -- should fit in one session, but if it doesn't, use the `--only=<phase>` flag below to run one phase per session and click **Save Version** between phases -- `run_ablation.py` skips phases that already finished, so re-running the whole notebook after a restart just picks up where it left off. Phase keys: `seed`, `base_koma`, `koma_master`, `koma_verification`, `koma_rag_full`.

In [ ]:
import os

REPO_URL = "https://github.com/hasnain1241/KoMA-RAG.git"
BRANCH = "KoMA-V2"
REPO_DIR = "/kaggle/working/KoMA-RAG"

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull
%cd {REPO_DIR}

# Path to the isolated Python 3.9 venv set up in the next cell. Every later
# `!...` call uses this instead of the kernel's own `python`/`pip`.
VENV_PY = "/kaggle/temp/venv/bin/python"

In [ ]:
# Kaggle's kernel Python is 3.12+, but requirements.txt pins numpy==1.24.3 /
# gymnasium==0.28.1 (this repo's original target: Python 3.9), which have no
# prebuilt wheels past Python 3.11 and fail to build from source here (a
# real run of this notebook hit exactly this and every phase crashed in ~2s
# with a broken numpy import). Fix: use uv to fetch an isolated Python 3.9
# + venv under /kaggle/temp (ephemeral, won't bloat Save Version output),
# same as what worked locally -- don't fight the kernel's own interpreter.
!pip install -q uv
!uv python install 3.9
!uv venv --python 3.9 /kaggle/temp/venv

# CPU-only torch first: sentence-transformers pulls in torch, and with the
# accelerator left off there's no GPU here, so skip the ~3GB CUDA wheel set.
!uv pip install --python {VENV_PY} torch --index-url https://download.pytorch.org/whl/cpu
!uv pip install --python {VENV_PY} -r requirements.txt python-dotenv

!{VENV_PY} --version

In [ ]:
# Mistral key from Kaggle Secrets (never paste the key directly into a cell).
from kaggle_secrets import UserSecretsClient
os.environ["MISTRAL_API_KEY"] = UserSecretsClient().get_secret("MISTRAL_API_KEY")

# Kaggle has no display server; highway_env/pygame need a dummy SDL driver
# to render frames off-screen (see highway_env/envs/common/graphics.py).
os.environ["SDL_VIDEODRIVER"] = "dummy"

In [ ]:
# Full unattended run (all phases, aggregates at the end).
# If you're worried about the session time limit, comment this out and
# use the per-phase cell below instead.
!{VENV_PY} run_ablation.py

## Alternative: one phase per session

Run this cell instead of the one above, changing `PHASE` each session, then click **Save Version** so `/kaggle/working/KoMA-RAG/result/` is preserved before the session ends.

In [ ]:
PHASE = "seed"  # seed -> base_koma -> koma_master -> koma_verification -> koma_rag_full
!{VENV_PY} run_ablation.py --only={PHASE}

In [ ]:
# Once all 4 ablation configs have finished (in any number of sessions),
# rebuild the mean +/- std table without re-running anything:
!{VENV_PY} run_ablation.py --aggregate-only
print(open("result/ablation_summary.md").read())